# 15. Multi-Target DrugCentral Exploration

This notebook collects DrugCentral evidence for the project targets.

Goal:
- Pull target-drug activity records from DrugCentral.
- Pull target component metadata.
- Pull structure metadata for drugs linked to the targets.
- Pull indication, off-label use, and contraindication relationships for those structures.
- Save raw API responses and processed CSV summaries.

This is the final source-exploration notebook before building the merged multi-target knowledge base.

In [ ]:
import json
import re
import time
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw" / "drugcentral"
PROCESSED_DIR = DATA_DIR / "processed"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

RECOMMENDATIONS_FILE = PROCESSED_DIR / "multi_target_drug_recommendations.csv"
TARGETS_FILE = PROCESSED_DIR / "multi_target_chembl_targets.csv"

print("Project root:", PROJECT_ROOT)
print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

## 1. Load Project Data

The recommendation file comes from notebook `09_multi_target_chembl_exploration.ipynb`.

In [ ]:
if not RECOMMENDATIONS_FILE.exists():
    raise FileNotFoundError(
        f"Missing {RECOMMENDATIONS_FILE}. Run notebook 09_multi_target_chembl_exploration.ipynb first."
    )

recommendations_df = pd.read_csv(RECOMMENDATIONS_FILE)
print("Recommendation rows:", len(recommendations_df))
display(recommendations_df.head())

if TARGETS_FILE.exists():
    targets_df = pd.read_csv(TARGETS_FILE)
else:
    targets_df = recommendations_df[["target_symbol", "target_display_name", "target_full_name", "target_chembl_id"]].drop_duplicates()

print("Target rows:", len(targets_df))
display(targets_df)

In [ ]:
TARGET_SYMBOLS = ["EGFR", "ERBB2", "BRAF", "ALK", "KRAS", "VEGFA", "MET", "PIK3CA"]
DRUGCENTRAL_API_BASE = "https://uxn2ycvimg.us-east-2.awsapprunner.com"

# Limit is only for the expensive structure/relationship calls. Increase later if you want a deeper pull.
MAX_STRUCTURES_PER_TARGET = 80

print("Targets configured:", TARGET_SYMBOLS)
print("Max structures per target:", MAX_STRUCTURES_PER_TARGET)

## 2. Helper Functions

In [ ]:
def normalize_name(value):
    """Normalize drug names for safer matching across datasets."""
    if pd.isna(value):
        return ""
    return re.sub(r"[^A-Z0-9]+", "", str(value).upper())


def safe_join(values, limit=8):
    cleaned = [str(value).strip() for value in values if pd.notna(value) and str(value).strip()]
    unique_values = list(dict.fromkeys(cleaned))
    return " | ".join(unique_values[:limit])


def get_json(path, retries=3, pause=1.5):
    """GET JSON from a DrugCentral API path with simple retries."""
    url = f"{DRUGCENTRAL_API_BASE}{path}"
    for attempt in range(retries):
        try:
            response = requests.get(url, timeout=(10, 90))
            if response.status_code == 200:
                return response.json()
            if response.status_code == 404:
                return []
            print(f"  {path} attempt {attempt + 1}: HTTP {response.status_code}; retrying")
        except requests.exceptions.RequestException as error:
            print(f"  {path} attempt {attempt + 1}: {type(error).__name__}; retrying")
        time.sleep(pause)
    return []


def get_project_drug_norms(target_symbol):
    subset = recommendations_df[recommendations_df["target_symbol"] == target_symbol]
    return {normalize_name(value) for value in subset["drug_name"].dropna().unique()}


def coerce_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, dict):
        return [value]
    return []

## 3. Pull DrugCentral Target Activity Records

DrugCentral activity records connect target genes to drugs or compounds.

In [ ]:
activity_raw = {}
target_component_raw = {}

for target_symbol in TARGET_SYMBOLS:
    print(f"Fetching DrugCentral target data for {target_symbol}")
    activity_rows = coerce_list(get_json(f"/act_table_full/gene/{target_symbol}"))
    component_rows = coerce_list(get_json(f"/target_component/gene/{target_symbol}"))

    activity_raw[target_symbol] = activity_rows
    target_component_raw[target_symbol] = component_rows

    print(f"  activity rows: {len(activity_rows)}")
    print(f"  component rows: {len(component_rows)}")
    time.sleep(0.25)

activity_raw_file = RAW_DIR / "multi_target_drugcentral_activity_raw.json"
component_raw_file = RAW_DIR / "multi_target_drugcentral_target_component_raw.json"

with activity_raw_file.open("w") as f:
    json.dump(activity_raw, f, indent=2)

with component_raw_file.open("w") as f:
    json.dump(target_component_raw, f, indent=2)

print("Saved:", activity_raw_file)
print("Saved:", component_raw_file)

In [ ]:
activity_records = []
component_records = []

for target_symbol, rows in activity_raw.items():
    project_drugs = get_project_drug_norms(target_symbol)
    for row in rows:
        record = dict(row)
        record["target_symbol"] = target_symbol
        record["source"] = "DrugCentral"
        activity_records.append(record)

for target_symbol, rows in target_component_raw.items():
    for row in rows:
        record = dict(row)
        record["target_symbol"] = target_symbol
        record["source"] = "DrugCentral"
        component_records.append(record)

activity_df = pd.DataFrame(activity_records)
target_component_df = pd.DataFrame(component_records)

print("Activity dataframe rows:", len(activity_df))
print("Target component dataframe rows:", len(target_component_df))
display(activity_df.head())

## 4. Pull Structure and Relationship Records

The activity endpoint gives DrugCentral structure IDs. The structure and relationship endpoints add drug names, indication relationships, off-label use, and contraindications.

In [ ]:
if "struct_id" in activity_df.columns:
    target_structures_df = activity_df[["target_symbol", "struct_id"]].dropna().drop_duplicates().copy()
    target_structures_df["struct_id"] = target_structures_df["struct_id"].astype(int)
else:
    target_structures_df = pd.DataFrame(columns=["target_symbol", "struct_id"])

limited_target_structures = []
for target_symbol, group in target_structures_df.groupby("target_symbol"):
    limited_group = group.sort_values("struct_id").head(MAX_STRUCTURES_PER_TARGET)
    limited_target_structures.append(limited_group)

if limited_target_structures:
    limited_target_structures_df = pd.concat(limited_target_structures, ignore_index=True)
else:
    limited_target_structures_df = pd.DataFrame(columns=["target_symbol", "struct_id"])

unique_struct_ids = sorted(limited_target_structures_df["struct_id"].dropna().astype(int).unique().tolist())

print("Unique structure IDs in activity data:", target_structures_df["struct_id"].nunique() if not target_structures_df.empty else 0)
print("Unique structure IDs selected for enrichment:", len(unique_struct_ids))
display(limited_target_structures_df.head())

In [ ]:
structures_raw = {}
relationships_raw = {}

for index, struct_id in enumerate(unique_struct_ids, start=1):
    structures_raw[str(struct_id)] = coerce_list(get_json(f"/structures/id/{struct_id}"))
    relationships_raw[str(struct_id)] = coerce_list(get_json(f"/omop_relationship/struct_id/{struct_id}"))

    if index % 25 == 0 or index == len(unique_struct_ids):
        print(f"Fetched {index}/{len(unique_struct_ids)} structure IDs")
    time.sleep(0.05)

structures_raw_file = RAW_DIR / "multi_target_drugcentral_structures_raw.json"
relationships_raw_file = RAW_DIR / "multi_target_drugcentral_relationships_raw.json"

with structures_raw_file.open("w") as f:
    json.dump(structures_raw, f, indent=2)

with relationships_raw_file.open("w") as f:
    json.dump(relationships_raw, f, indent=2)

print("Saved:", structures_raw_file)
print("Saved:", relationships_raw_file)

In [ ]:
structure_records = []
relationship_records = []

struct_to_targets = (
    limited_target_structures_df.groupby("struct_id")["target_symbol"]
    .apply(lambda values: sorted(set(values)))
    .to_dict()
    if not limited_target_structures_df.empty
    else {}
)

for struct_id, rows in structures_raw.items():
    for row in rows:
        record = dict(row)
        record["struct_id"] = int(struct_id)
        record["target_symbols"] = " | ".join(struct_to_targets.get(int(struct_id), []))
        record["source"] = "DrugCentral"
        structure_records.append(record)

for struct_id, rows in relationships_raw.items():
    for row in rows:
        record = dict(row)
        record["struct_id"] = int(struct_id)
        record["target_symbols"] = " | ".join(struct_to_targets.get(int(struct_id), []))
        record["source"] = "DrugCentral"
        relationship_records.append(record)

structures_df = pd.DataFrame(structure_records)
relationships_df = pd.DataFrame(relationship_records)

print("Structure rows:", len(structures_df))
print("Relationship rows:", len(relationships_df))
display(structures_df.head())
display(relationships_df.head())

## 5. Build Processed Activity and Indication Tables

In [ ]:
if structures_df.empty:
    structure_names_df = pd.DataFrame(columns=["struct_id", "drugcentral_name", "cas_reg_no", "fda_labels", "no_formulations"])
else:
    name_columns = [column for column in ["id", "name", "cas_reg_no", "fda_labels", "no_formulations"] if column in structures_df.columns]
    structure_names_df = structures_df[name_columns].drop_duplicates().copy()
    if "id" in structure_names_df.columns:
        structure_names_df = structure_names_df.rename(columns={"id": "struct_id"})
    if "name" in structure_names_df.columns:
        structure_names_df = structure_names_df.rename(columns={"name": "drugcentral_name"})

if activity_df.empty:
    drugcentral_activity_df = pd.DataFrame(columns=[
        "target_symbol", "gene", "struct_id", "drugcentral_name", "normalised_drug_name",
        "matches_project_drug", "act_type", "relation", "act_value", "act_unit",
        "action_type", "moa", "source",
    ])
else:
    drugcentral_activity_df = activity_df.merge(structure_names_df, on="struct_id", how="left") if "struct_id" in activity_df.columns else activity_df.copy()
    if "drugcentral_name" not in drugcentral_activity_df.columns:
        drugcentral_activity_df["drugcentral_name"] = drugcentral_activity_df.get("name", "")

    drugcentral_activity_df["normalised_drug_name"] = drugcentral_activity_df["drugcentral_name"].apply(normalize_name)
    drugcentral_activity_df["matches_project_drug"] = drugcentral_activity_df.apply(
        lambda row: row["normalised_drug_name"] in get_project_drug_norms(row["target_symbol"]),
        axis=1,
    )
    drugcentral_activity_df["source"] = "DrugCentral"

activity_columns = [
    "target_symbol", "gene", "accession", "swissprot", "target_name", "target_class", "tdl",
    "struct_id", "drugcentral_name", "normalised_drug_name", "matches_project_drug",
    "act_type", "relation", "act_value", "act_unit", "act_source", "act_comment",
    "action_type", "moa", "cas_reg_no", "fda_labels", "no_formulations", "source",
]
activity_columns = [column for column in activity_columns if column in drugcentral_activity_df.columns]
drugcentral_activity_df = drugcentral_activity_df[activity_columns]

activity_file = PROCESSED_DIR / "multi_target_drugcentral_activity.csv"
drugcentral_activity_df.to_csv(activity_file, index=False)

print("Saved:", activity_file)
print("Rows:", len(drugcentral_activity_df))
display(drugcentral_activity_df.head())

In [ ]:
if relationships_df.empty:
    drugcentral_indications_df = pd.DataFrame(columns=[
        "target_symbols", "struct_id", "drugcentral_name", "relationship_name", "concept_name",
        "snomed_full_name", "snomed_conceptid", "umls_cui", "source",
    ])
else:
    drugcentral_indications_df = relationships_df.merge(structure_names_df, on="struct_id", how="left")
    if "drugcentral_name" not in drugcentral_indications_df.columns:
        drugcentral_indications_df["drugcentral_name"] = drugcentral_indications_df.get("name", "")
    drugcentral_indications_df["source"] = "DrugCentral"

    indication_columns = [
        "target_symbols", "struct_id", "drugcentral_name", "relationship_name", "concept_name",
        "snomed_full_name", "snomed_conceptid", "umls_cui", "source",
    ]
    for column in indication_columns:
        if column not in drugcentral_indications_df.columns:
            drugcentral_indications_df[column] = ""
    drugcentral_indications_df = drugcentral_indications_df[indication_columns]

indications_file = PROCESSED_DIR / "multi_target_drugcentral_indications.csv"
drugcentral_indications_df.to_csv(indications_file, index=False)

print("Saved:", indications_file)
print("Rows:", len(drugcentral_indications_df))
display(drugcentral_indications_df.head())

## 6. Build DrugCentral Summary and Coverage Tables

In [ ]:
if drugcentral_indications_df.empty:
    relationship_summary_df = pd.DataFrame(columns=[
        "struct_id", "drugcentral_name", "indication_count", "off_label_count",
        "contraindication_count", "top_indications",
    ])
else:
    relationship_summary_df = drugcentral_indications_df.groupby(
        ["struct_id", "drugcentral_name"], dropna=False
    ).agg(
        indication_count=("relationship_name", lambda s: int((s.astype(str).str.lower() == "indication").sum())),
        off_label_count=("relationship_name", lambda s: int((s.astype(str).str.lower() == "off-label use").sum())),
        contraindication_count=("relationship_name", lambda s: int((s.astype(str).str.lower() == "contraindication").sum())),
        top_indications=("concept_name", safe_join),
    ).reset_index()

if drugcentral_activity_df.empty:
    activity_summary_df = pd.DataFrame(columns=[
        "target_symbol", "struct_id", "drugcentral_name", "activity_count", "min_activity_value",
        "activity_types", "activity_sources", "matches_project_drug",
    ])
else:
    activity_summary_df = drugcentral_activity_df.groupby(
        ["target_symbol", "struct_id", "drugcentral_name"], dropna=False
    ).agg(
        activity_count=("struct_id", "size"),
        min_activity_value=("act_value", "min") if "act_value" in drugcentral_activity_df.columns else ("struct_id", "size"),
        activity_types=("act_type", safe_join) if "act_type" in drugcentral_activity_df.columns else ("struct_id", lambda s: ""),
        activity_sources=("act_source", safe_join) if "act_source" in drugcentral_activity_df.columns else ("struct_id", lambda s: ""),
        matches_project_drug=("matches_project_drug", "max"),
    ).reset_index()

drugcentral_summary_df = activity_summary_df.merge(
    relationship_summary_df,
    on=["struct_id", "drugcentral_name"],
    how="left",
)

for column in ["indication_count", "off_label_count", "contraindication_count"]:
    if column in drugcentral_summary_df.columns:
        drugcentral_summary_df[column] = drugcentral_summary_df[column].fillna(0).astype(int)
if "top_indications" in drugcentral_summary_df.columns:
    drugcentral_summary_df["top_indications"] = drugcentral_summary_df["top_indications"].fillna("")

drugcentral_summary_df["has_indication_evidence"] = drugcentral_summary_df.get("indication_count", 0) > 0
drugcentral_summary_df["has_off_label_evidence"] = drugcentral_summary_df.get("off_label_count", 0) > 0
drugcentral_summary_df["has_contraindication_evidence"] = drugcentral_summary_df.get("contraindication_count", 0) > 0

drugcentral_summary_file = PROCESSED_DIR / "multi_target_drugcentral_summary.csv"
drugcentral_summary_df.to_csv(drugcentral_summary_file, index=False)

print("Saved:", drugcentral_summary_file)
print("Rows:", len(drugcentral_summary_df))
display(drugcentral_summary_df.head(30))

In [ ]:
base_targets_df = pd.DataFrame({"target_symbol": TARGET_SYMBOLS})

if drugcentral_activity_df.empty:
    coverage_df = base_targets_df.copy()
    coverage_df["drugcentral_activity_rows"] = 0
    coverage_df["drugcentral_unique_drug_count"] = 0
    coverage_df["project_drug_match_count"] = 0
    coverage_df["drugcentral_drugs_with_indications"] = 0
else:
    activity_coverage_df = drugcentral_activity_df.groupby("target_symbol").agg(
        drugcentral_activity_rows=("target_symbol", "size"),
        drugcentral_unique_drug_count=("drugcentral_name", "nunique"),
        project_drug_match_count=("matches_project_drug", "sum"),
    ).reset_index()

    if drugcentral_summary_df.empty:
        indication_coverage_df = pd.DataFrame(columns=["target_symbol", "drugcentral_drugs_with_indications"])
    else:
        indication_coverage_df = drugcentral_summary_df.groupby("target_symbol").agg(
            drugcentral_drugs_with_indications=("has_indication_evidence", "sum"),
            drugcentral_drugs_with_off_label=("has_off_label_evidence", "sum"),
            drugcentral_drugs_with_contraindications=("has_contraindication_evidence", "sum"),
        ).reset_index()

    coverage_df = (
        base_targets_df
        .merge(activity_coverage_df, on="target_symbol", how="left")
        .merge(indication_coverage_df, on="target_symbol", how="left")
    )

count_columns = [
    "drugcentral_activity_rows", "drugcentral_unique_drug_count", "project_drug_match_count",
    "drugcentral_drugs_with_indications", "drugcentral_drugs_with_off_label",
    "drugcentral_drugs_with_contraindications",
]
for column in count_columns:
    if column not in coverage_df.columns:
        coverage_df[column] = 0
    coverage_df[column] = coverage_df[column].fillna(0).astype(int)

coverage_df["source_status"] = coverage_df["drugcentral_activity_rows"].apply(lambda value: "working" if value > 0 else "no activity found")

coverage_file = PROCESSED_DIR / "multi_target_drugcentral_coverage_summary.csv"
coverage_df.to_csv(coverage_file, index=False)

print("Saved:", coverage_file)
display(coverage_df)

## 7. Final Check

After running this notebook, the source-exploration stage is complete. The next stage is to merge all processed outputs into one multi-target knowledge base.

In [ ]:
print("DrugCentral Multi-Target Exploration Complete")
print("=" * 70)
print("Targets:", len(TARGET_SYMBOLS))
print("Activity rows:", len(drugcentral_activity_df))
print("Indication/relationship rows:", len(drugcentral_indications_df))
print("Summary rows:", len(drugcentral_summary_df))
print("Coverage rows:", len(coverage_df))
print("Files created:")
print("-", activity_raw_file)
print("-", component_raw_file)
print("-", structures_raw_file)
print("-", relationships_raw_file)
print("-", activity_file)
print("-", indications_file)
print("-", drugcentral_summary_file)
print("-", coverage_file)

display(coverage_df)